# RK1828 YOLO26 姿态估计 & 实例分割 Demo


两个基于 **YOLO26** 的下游任务，都是官方 `rknn3-model-zoo` 的预编译 C++ demo：

- **姿态估计**（pose）：检测人体 + 17 个关键点（n/s/m 三个规格，i8/fp16 两种量化；
  例外：**m 规格只有 i8**——官方 SDK 包里的 m-pose fp16 权重是损坏的，实为 .rknn 文件的副本，故未收录）
- **实例分割**（segment）：检测框 + 像素级掩码（n/s/m，i8/fp16）

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型和 demo 程序（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 姿态估计 |
| 4 | 实例分割 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("yolo26-pose-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/yolo26_pose/demo_x86_64.tar.gz"),
    ("yolo26-pose-rknn3-rk1828", "model/coco_80_labels_list.txt", "demos/yolo26_pose/model/coco_80_labels_list.txt"),
    ("yolo26-pose-rknn3-rk1828", "model/little_boy.jpg", "demos/yolo26_pose/model/little_boy.jpg"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26m-pose_i8.rknn", "demos/yolo26_pose/model/yolo26m-pose_i8.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26m-pose_i8.weight", "demos/yolo26_pose/model/yolo26m-pose_i8.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_fp16.rknn", "demos/yolo26_pose/model/yolo26n-pose_fp16.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_fp16.weight", "demos/yolo26_pose/model/yolo26n-pose_fp16.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_i8.rknn", "demos/yolo26_pose/model/yolo26n-pose_i8.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_i8.weight", "demos/yolo26_pose/model/yolo26n-pose_i8.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_fp16.rknn", "demos/yolo26_pose/model/yolo26s-pose_fp16.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_fp16.weight", "demos/yolo26_pose/model/yolo26s-pose_fp16.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_i8.rknn", "demos/yolo26_pose/model/yolo26s-pose_i8.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_i8.weight", "demos/yolo26_pose/model/yolo26s-pose_i8.weight"),
    ("yolo26-segment-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/yolo26_segment/demo_x86_64.tar.gz"),
    ("yolo26-segment-rknn3-rk1828", "model/bus.jpg", "demos/yolo26_segment/model/bus.jpg"),
    ("yolo26-segment-rknn3-rk1828", "model/coco_80_labels_list.txt", "demos/yolo26_segment/model/coco_80_labels_list.txt"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_fp16.rknn", "demos/yolo26_segment/model/yolo26m-seg_fp16.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_fp16.weight", "demos/yolo26_segment/model/yolo26m-seg_fp16.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_i8.rknn", "demos/yolo26_segment/model/yolo26m-seg_i8.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_i8.weight", "demos/yolo26_segment/model/yolo26m-seg_i8.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_fp16.rknn", "demos/yolo26_segment/model/yolo26n-seg_fp16.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_fp16.weight", "demos/yolo26_segment/model/yolo26n-seg_fp16.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_i8.rknn", "demos/yolo26_segment/model/yolo26n-seg_i8.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_i8.weight", "demos/yolo26_segment/model/yolo26n-seg_i8.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_fp16.rknn", "demos/yolo26_segment/model/yolo26s-seg_fp16.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_fp16.weight", "demos/yolo26_segment/model/yolo26s-seg_fp16.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_i8.rknn", "demos/yolo26_segment/model/yolo26s-seg_i8.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_i8.weight", "demos/yolo26_segment/model/yolo26s-seg_i8.weight"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/yolo26_pose/demo_x86_64.tar.gz", "demos/yolo26_pose")
extract_tarball("demos/yolo26_segment/demo_x86_64.tar.gz", "demos/yolo26_segment")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 姿态估计（17 关键点）

参考成绩：yolo26m-pose i8 约 152ms/帧，示例图上人体置信度 0.93。

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_size = widgets.Dropdown(options=["n", "s", "m"], value="m", description="规格:")
w_quant = widgets.Dropdown(options=["i8", "fp16"], description="量化:")
display(w_size, w_quant)
print("选好后继续往下运行 ▼")

In [ ]:
import os

DEMO = "demos/yolo26_pose"
m = f"yolo26{w_size.value}-pose_{w_quant.value}"
if not os.path.exists(f"{DEMO}/model/{m}.weight"):
    print(f"⚠️ {m} 不存在：m 规格的 fp16 权重在官方 SDK 中损坏，请改用 i8 或 n/s 规格")
else:
    before = snapshot(DEMO)
    run_demo([f"./rknn_yolo26_pose_demo", f"model/{m}.rknn", f"model/{m}.weight",
              "model/little_boy.jpg", "0x1"], DEMO)
    show_new_images(DEMO, before)

## 4. 实例分割（检测框 + 像素掩码）

参考成绩：yolo26m-seg i8 约 261ms/帧，COCO 示例图检出 4 个人 + 1 辆巴士。

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_size2 = widgets.Dropdown(options=["n", "s", "m"], value="m", description="规格:")
w_quant2 = widgets.Dropdown(options=["i8", "fp16"], description="量化:")
display(w_size2, w_quant2)
print("选好后继续往下运行 ▼")

In [ ]:
DEMO = "demos/yolo26_segment"
m = f"yolo26{w_size2.value}-seg_{w_quant2.value}"
before = snapshot(DEMO)
run_demo([f"./rknn_yolo26_segment_demo", f"model/{m}.rknn", f"model/{m}.weight",
          "model/bus.jpg", "0x1"], DEMO)
show_new_images(DEMO, before)

## 小结

- 两个 demo 的模型都是**全套官方转换产物**，后处理（含 NMS）由 C++ demo 完成。
- 可换 n/s（更小更快）规格对比精度与耗时。